# Домашнее задание 2



In [27]:
import pandas as pd
import re

## 1. Загрузка датасета

Загружаем датасет `train.csv` с данными о выживаемости пассажиров Титаника.

In [28]:
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Основная информация о датасете

Выведем информацию о типах данных, пропусках, статистические характеристики числовых колонок.

In [29]:
print('=== Размер датасета ===')
print(f'Строк: {df.shape[0]}, Колонок: {df.shape[1]}')

=== Размер датасета ===
Строк: 891, Колонок: 12


In [30]:
print('=== Типы данных ===')
df.info()

=== Типы данных ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [31]:
print('=== Количество пропусков по колонкам ===')
df.isnull().sum()

=== Количество пропусков по колонкам ===


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [32]:
print('=== Статистика числовых колонок ===')
df.describe()

=== Статистика числовых колонок ===


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


## 3. Процент выживаемости по классам пассажиров

Посчитаем долю выживших (Survived=1) в каждом классе (Pclass).

In [33]:
survival_rate = df.groupby('Pclass')['Survived'].mean() * 100
survival_rate = survival_rate.rename('Выживаемость (%)')
print('Процент выживаемости по классам:')
survival_rate.round(2)

Процент выживаемости по классам:


Pclass
1    62.96
2    47.28
3    24.24
Name: Выживаемость (%), dtype: float64

## 4. Самое популярное мужское и женское имя на корабле

Имена в датасете хранятся в формате `"Фамилия, Титул. Имя Отчество"`. Извлечём имена с помощью регулярного выражения.

In [34]:
def extract_first_name(full_name):
    if 'Mrs.' in full_name:
        paren_match = re.search(r'\((\w+)', full_name)
        if paren_match:
            return paren_match.group(1)
    match = re.search(r',\s+\w+\.\s+(\w+)', full_name)
    if match:
        return match.group(1)
    return None

df['FirstName'] = df['Name'].apply(extract_first_name)

# Определяем пол через колонку Sex
male_names = df[df['Sex'] == 'male']['FirstName']
female_names = df[df['Sex'] == 'female']['FirstName']

top_male = male_names.value_counts().idxmax()
top_female = female_names.value_counts().idxmax()

print(f'Самое популярное мужское имя: {top_male} ({male_names.value_counts().max()} раз)')
print(f'Самое популярное женское имя: {top_female} ({female_names.value_counts().max()} раз)')

Самое популярное мужское имя: William (35 раз)
Самое популярное женское имя: Anna (15 раз)


## 5. Самое популярное мужское и женское имя в каждом классе

In [35]:
for pclass in sorted(df['Pclass'].unique()):
    class_df = df[df['Pclass'] == pclass]
    top_m = class_df[class_df['Sex'] == 'male']['FirstName'].value_counts().idxmax()
    top_f = class_df[class_df['Sex'] == 'female']['FirstName'].value_counts().idxmax()
    print(f'Класс {pclass}: мужское — {top_m}, женское — {top_f}')

Класс 1: мужское — William, женское — Elizabeth
Класс 2: мужское — William, женское — Elizabeth
Класс 3: мужское — William, женское — Anna


## 6. Пассажиры старше 44 лет

In [36]:
older_than_44 = df[df['Age'] > 44]
print(f'Количество пассажиров старше 44 лет: {len(older_than_44)}')
older_than_44

Количество пассажиров старше 44 лет: 115


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Timothy
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,Elizabeth
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S,Mary
33,34,0,2,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.5000,NaN,S,Edward
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C,Myna
...,...,...,...,...,...,...,...,...,...,...,...,...,...
857,858,1,1,"Daly, Mr. Peter Denis",male,51.0,0,0,113055,26.5500,E17,S,Peter
862,863,1,1,"Swift, Mrs. Frederick Joel (Margaret Welles Ba...",female,48.0,0,0,17466,25.9292,D17,S,Margaret
871,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,52.5542,D35,S,Sallie
873,874,0,3,"Vander Cruyssen, Mr. Victor",male,47.0,0,0,345765,9.0000,NaN,S,Victor


## 7. Пассажиры моложе 44 лет мужского пола

In [37]:
young_males = df[(df['Age'] < 44) & (df['Sex'] == 'male')]
print(f'Количество мужчин моложе 44 лет: {len(young_males)}')
young_males

Количество мужчин моложе 44 лет: 368


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.250,NaN,S,Owen
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.050,NaN,S,William
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.075,NaN,S,Gosta
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.050,NaN,S,William
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.275,NaN,S,Anders
...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,884,0,2,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.500,NaN,S,Frederick
884,885,0,3,"Sutehall, Mr. Henry Jr",male,25.0,0,0,SOTON/OQ 392076,7.050,NaN,S,Henry
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.000,NaN,S,Juozas
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.000,C148,C,Karl


## 8. Количество n-местных кабин

Посчитаем, сколько кабин было занято 2, 3, 4 и более пассажирами.

In [38]:
cabin_df = df[df['Cabin'].notna()].copy()

cabin_df['CabinPrimary'] = cabin_df['Cabin'].str.split().str[0]

cabin_counts = cabin_df.groupby('CabinPrimary').size()

n_person_cabins = cabin_counts.value_counts().sort_index()
n_person_cabins.index.name = 'Кол-во пассажиров в каюте'
n_person_cabins.name = 'Кол-во таких кают'
print('Распределение кают по числу жильцов:')
n_person_cabins

Распределение кают по числу жильцов:


Кол-во пассажиров в каюте
1    99
2    37
3     5
4     4
Name: Кол-во таких кают, dtype: int64

## 9. Пассажиры без родственников на борту

Пассажиры без родственников - те, у кого `SibSp == 0` (нет братьев/сестёр/супругов) **и** `Parch == 0` (нет родителей/детей).

In [39]:
alone = df[(df['SibSp'] == 0) & (df['Parch'] == 0)]
print(f'Количество пассажиров без родственников на борту: {len(alone)}')
print(f'Это {len(alone) / len(df) * 100:.1f}% от всех пассажиров')

Количество пассажиров без родственников на борту: 537
Это 60.3% от всех пассажиров
